## scATAC preprocessing

### Setup libraries

In [ ]:
import numpy as np
import snapatac2 as snap
import sys, os
from collections import defaultdict
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
chrom_sizes_dict = defaultdict(int)

print('Loading chromosomes', file=sys.stderr)
with open("references/mm39.chrom.sizes", "r") as fh:
    for line in fh:
        if not ("random" in line.strip().split("\t")[0] or "Un" in line.strip().split("\t")[0]):
            chrom_sizes_dict[line.strip().split("\t")[0]] = int(line.strip().split("\t")[1])

chrom_sizes_dict.keys()

### Create snapATAC2 object and calculate basic QCs.

This assumes you have run the download ATAC fragments script.

Then we apply a very minimal filter of 100 fragments per barcode and calculate some basic QCs per barcode to plot. 

In [ ]:
ssecpkr_list = ["IGVFFI5731CAQI","IGVFFI5042DAMZ","IGVFFI0753YWZB","IGVFFI6974VPAB","IGVFFI8189LVJD"]

fragment_files = [f"data/{ssecpkr}.bed_sorted.gz" for ssecpkr in ssecpkr_list]
output_names = [f"data/{ssecpkr}_atac.h5ad" for ssecpkr in ssecpkr_list]
adatas = snap.pp.import_fragments(
    fragment_file=fragment_files,
    file=output_names,
    chrom_sizes=chrom_sizes_dict,
    sorted_by_barcode=True,
    min_num_fragments=100,
    shift_left=4,
    shift_right=-4,
    n_jobs=4   
)

In [ ]:
snap.metrics.tsse(adatas, "references/mm39.genes.gtf")
adatas

In [ ]:
snap.pp.add_tile_matrix(adatas)

In [ ]:
adataset = snap.AnnDataSet(
    adatas=[(f.filename.split('/')[-1].split('.h5ad')[0], f) for f in adatas],
    filename="data/atac_adataset.h5ad"
)
adataset

In [ ]:
adataset.obs["n_fragment"] = adataset.adatas.obs['n_fragment']
adataset.obs["frac_dup"] = adataset.adatas.obs['frac_dup']
adataset.obs["frac_mito"] = adataset.adatas.obs['frac_mito']
adataset.obs["tsse"] = adataset.adatas.obs['tsse']

In [ ]:
ob = pd.DataFrame({ 'n_fragments': adataset.obs["n_fragment"], 
                   'tsse': adataset.obs["tsse"],
                  'frac_dup': adataset.obs["frac_dup"],
                  'frac_mito': adataset.obs["frac_mito"],
                  'sample':adataset.obs["sample"],
                  'barcode': adataset.obs_names})
ob.to_csv("results/filt_atac_100frags.barcode_metrics.tsv", sep="\t", index=False)
adataset.close()

In [ ]:
adataset.close()

# Extra

In [ ]:
snap.pl.tsse(adataset, interactive=False)


In [ ]:
snap.pl.frag_size_distr(adataset, interactive=False)
